# Endpoint Process Anomaly Detection

Rank unusual endpoint processes with a multivariate distance model trained on a clean baseline.

**Safety and scope:** This project uses synthetic, non-sensitive telemetry for defensive analytics. It does not perform exploitation or execute malicious content.

## Goal

Implement Mahalanobis-distance anomaly scoring and evaluate precision among the highest-ranked processes.


## Setup

The notebook is deterministic, runs offline, and implements the core analytical method directly with NumPy and Pandas so the modeling logic remains inspectable.


In [1]:
import numpy as np
import pandas as pd

SEED = 42
rng = np.random.default_rng(SEED)
pd.set_option("display.width", 120)


## Steps

### 1. Generate process telemetry


In [2]:
normal_count = 700
suspicious_count = 55

normal = pd.DataFrame({
    "command_length": rng.normal(42, 13, normal_count).clip(5),
    "child_processes": rng.poisson(1.2, normal_count),
    "network_connections": rng.poisson(0.9, normal_count),
    "unsigned_binary": rng.binomial(1, 0.05, normal_count),
    "rare_path": rng.binomial(1, 0.04, normal_count),
    "encoded_marker": rng.binomial(1, 0.015, normal_count),
    "suspicious": 0,
})
suspicious = pd.DataFrame({
    "command_length": rng.normal(145, 28, suspicious_count).clip(30),
    "child_processes": rng.poisson(4.8, suspicious_count),
    "network_connections": rng.poisson(6.0, suspicious_count),
    "unsigned_binary": rng.binomial(1, 0.72, suspicious_count),
    "rare_path": rng.binomial(1, 0.68, suspicious_count),
    "encoded_marker": rng.binomial(1, 0.58, suspicious_count),
    "suspicious": 1,
})
process_events = pd.concat([normal, suspicious], ignore_index=True)
process_events = process_events.iloc[rng.permutation(len(process_events))].reset_index(drop=True)

print("Process count:", len(process_events))
print("Suspicious rate:", round(process_events["suspicious"].mean(), 3))
print(process_events.sample(6, random_state=SEED).round(2).to_string(index=False))


Process count: 755
Suspicious rate: 0.073
 command_length  child_processes  network_connections  unsigned_binary  rare_path  encoded_marker  suspicious
          47.02                2                    1                0          0               0           0
          46.48                1                    1                0          0               0           0
          18.20                0                    5                0          0               0           0
          28.06                1                    0                0          1               0           0
          50.65                0                    1                0          0               0           0
         117.21                4                    9                1          0               1           1


### 2. Fit a clean-baseline anomaly model


In [3]:
feature_names = [column for column in process_events.columns if column != "suspicious"]
baseline = process_events[process_events["suspicious"] == 0][feature_names].to_numpy(float)
all_values = process_events[feature_names].to_numpy(float)

baseline_mean = baseline.mean(axis=0)
covariance = np.cov(baseline, rowvar=False) + np.eye(len(feature_names)) * 0.05
inverse_covariance = np.linalg.pinv(covariance)
centered = all_values - baseline_mean
anomaly_score = np.sqrt(np.einsum("ij,jk,ik->i", centered, inverse_covariance, centered))

ranked_processes = process_events.copy()
ranked_processes["anomaly_score"] = anomaly_score
ranked_processes = ranked_processes.sort_values("anomaly_score", ascending=False)
review_budget = suspicious_count
precision_at_budget = ranked_processes.head(review_budget)["suspicious"].mean()
baseline_p99 = float(np.quantile(anomaly_score[process_events["suspicious"].to_numpy() == 0], 0.99))

print("99th percentile clean-baseline score:", round(baseline_p99, 3))
print("Precision at review budget:", round(precision_at_budget, 3))
print("\nHighest-ranked processes:")
print(ranked_processes.head(10).round(3).to_string(index=False))


99th percentile clean-baseline score: 4.612
Precision at review budget: 1.0

Highest-ranked processes:
 command_length  child_processes  network_connections  unsigned_binary  rare_path  encoded_marker  suspicious  anomaly_score
        185.624                7                   14                1          1               1           1         18.938
        215.980                5                    4                0          1               1           1         15.068
        192.315                7                    7                1          1               0           1         14.803
        187.059                6                    5                1          1               1           1         13.988
        176.494                8                    5                1          1               1           1         13.935
        166.928                3                   10                0          0               1           1         13.869
        170.797       

## Checks


In [4]:
assert np.isfinite(anomaly_score).all()
assert precision_at_budget >= 0.85
assert ranked_processes["anomaly_score"].is_monotonic_decreasing
assert ranked_processes.head(10)["suspicious"].mean() >= 0.9
print("Checks passed: finite scores, high top-queue precision, and correctly ordered anomalies.")


Checks passed: finite scores, high top-queue precision, and correctly ordered anomalies.


## Next Steps

        - Build separate baselines by host role and operating system.
- Add signer, parent-child, prevalence, and command-line token features.
- Use analyst dispositions to tune the ranking threshold.
